In [7]:
import pandas as pd

# List of review files
files = [
    "reviews_0-250_masked.csv",
    "reviews_250-500_masked.csv",
    "reviews_500-750_masked.csv",
    "reviews_750-1250_masked.csv",
    "reviews_1250-end_masked.csv"
]

# Read and combine all files into one DataFrame
combined_reviews = pd.concat([pd.read_csv(file) for file in files], ignore_index=True)

# Print basic info
print("Combined dataset shape:", combined_reviews.shape)

# Display first few rows
combined_reviews.head()

Combined dataset shape: (285412, 19)


,Unnamed: 0.1,Unnamed: 0,rating,is_recommended,helpfulness,total_feedback_count,total_neg_feedback_count,total_pos_feedback_count,submission_time,review_text,review_title,skin_tone,eye_color,skin_type,hair_color,product_id,product_name,brand_name,price_usd
0,0,0,5,1.0,1.0,2,0,2,2023-02-01,I use this with the Nudestix “Citrus Clean Bal...,Taught me how to double cleanse!,NaN,brown,dry,black,P504322,Gentle Hydra-Gel Face Cleanser,NUDESTIX,19.0
1,1,1,1,0.0,NaN,0,0,0,2023-03-21,I bought this lip mask after reading the revie...,Disappointed,NaN,NaN,NaN,NaN,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.0
2,2,2,5,1.0,NaN,0,0,0,2023-03-21,My review title says it all! I get so excited ...,New Favorite Routine,light,brown,dry,blonde,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.0
3,3,3,5,1.0,NaN,0,0,0,2023-03-20,I’ve always loved this formula for a long time...,Can't go wrong with any of them,NaN,brown,combination,black,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.0
4,4,4,5,1.0,NaN,0,0,0,2023-03-20,"If you have dry cracked lips, this is a must h...",A must have !!!,light,hazel,combination,NaN,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.0


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# Copy dataset
df = combined_reviews.copy()

target = "rating"  # change if needed
df = df.dropna(subset=[target])

X = df.drop(columns=[target])
y = df[target]

categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
numeric_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

# Preprocessing
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_cols),
    ("cat", categorical_transformer, categorical_cols)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Fit Random Forest
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(n_estimators=100, random_state=42))
])

rf_pipeline.fit(X_train, y_train)

# Get feature names
feature_names = rf_pipeline.named_steps["preprocessor"].get_feature_names_out()

# Get importance
importances = rf_pipeline.named_steps["model"].feature_importances_

# Create DataFrame
feat_imp = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

print("\nTop 10 Random Forest Features:")
print(feat_imp.head(10))

# Linear Regression Coefficients
lr_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

lr_pipeline.fit(X_train, y_train)

coefficients = lr_pipeline.named_steps["model"].coef_

coef_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients
}).sort_values(by="Coefficient", key=abs, ascending=False)

print("\nTop 10 Linear Regression Features:")
print(coef_df.head(10))

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# Column names (adjust if needed)
text_col = "review_text"
rating_col = "rating"

df = combined_reviews.copy()

# Keep only high ratings (>= 4)
df_high = df[df[rating_col] >= 4]

# Drop missing text
df_high = df_high.dropna(subset=[text_col])

# TF-IDF
vectorizer = TfidfVectorizer(stop_words='english', max_features=1000)
X = vectorizer.fit_transform(df_high[text_col])

# Get words
words = np.array(vectorizer.get_feature_names_out())

# Sum TF-IDF scores across all high-rated reviews
scores = X.sum(axis=0).A1

# Get top 3 words
top_idx = np.argsort(scores)[-3:]

print("Top 3 Keywords for High Ratings:")
print(words[top_idx])

In [8]:
# Copy dataset
df = combined_reviews.copy()

target = "rating"
df = df.dropna(subset=[target])

In [10]:
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import CountVectorizer

# Filter high-rated reviews
df_high = df[df["rating"] >= 4].copy()

# Clean text
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Use the correct column: review_text
df_high["clean_text"] = df_high["review_text"].fillna("").apply(clean_text)

# Remove empty rows
df_high = df_high[df_high["clean_text"] != ""]

# Vectorize
vectorizer = CountVectorizer(
    stop_words="english",
    max_features=1000
)

X = vectorizer.fit_transform(df_high["clean_text"])

# Word frequencies
word_counts = np.asarray(X.sum(axis=0)).ravel()
words = vectorizer.get_feature_names_out()

word_freq = pd.DataFrame({
    "word": words,
    "count": word_counts
}).sort_values(by="count", ascending=False)

# Top keywords
print(word_freq.head(20))

           word   count
790        skin  295872
664     product  128630
511        love   99193
290        face   92350
934         use   91062
140    cleanser   71251
488        like   70901
525        mask   69757
244         dry   65185
937       using   63005
689      really   62515
372       great   49414
305        feel   47897
942          ve   46083
521      makeup   42902
455        just   39078
935        used   39019
307       feels   38817
807        soft   36801
214  definitely   34759
